In [19]:
# Install necessary packages
!pip install telethon python-dotenv




In [1]:
import os
from dotenv import load_dotenv
from telethon import TelegramClient
from telethon.tl.types import MessageMediaPhoto
from datetime import datetime
import asyncio
import json

# Load secrets
load_dotenv()


True

In [2]:
API_ID = int(os.getenv("TELEGRAM_API_ID"))  # or hardcode like: 123456
API_HASH = os.getenv("TELEGRAM_API_HASH")   # or hardcode like: 'abc123'
SESSION = os.getenv("TELEGRAM_SESSION", "session")

client = TelegramClient(SESSION, API_ID, API_HASH)

# Channels to scrape
CHANNELS = ['CheMed123', 'lobelia4cosmetics', 'tikvahpharma']
DATA_PATH = "data/raw/telegram_messages"


In [3]:
async def download_media(message, dest_folder):
    if isinstance(message.media, MessageMediaPhoto):
        path = await message.download_media(file=dest_folder)
        return os.path.basename(path)
    return None


In [4]:
async def scrape_channel(channel_username):
    await client.start()
    entity = await client.get_entity(channel_username)
    messages = []

    today = datetime.utcnow().strftime("%Y-%m-%d")
    img_folder = os.path.join(DATA_PATH, today, channel_username, "images")
    os.makedirs(img_folder, exist_ok=True)

    async for msg in client.iter_messages(entity, limit=None):
        img_filename = await download_media(msg, img_folder)
        messages.append({
            "message_id": msg.id,
            "date": msg.date.isoformat(),
            "text": msg.text,
            "has_image": bool(img_filename),
            "image_filename": img_filename
        })

    return messages



In [5]:
async def download_media(message, dest_folder):
    if isinstance(message.media, MessageMediaPhoto):
        path = await message.download_media(file=dest_folder)
        return os.path.basename(path)
    return None


In [6]:
async def scrape_channel(channel_username):
    await client.start()
    entity = await client.get_entity(channel_username)
    messages = []

    today = datetime.utcnow().strftime("%Y-%m-%d")
    img_folder = os.path.join(DATA_PATH, today, channel_username, "images")
    os.makedirs(img_folder, exist_ok=True)

    async for msg in client.iter_messages(entity, limit=None):
        img_filename = await download_media(msg, img_folder)
        messages.append({
            "message_id": msg.id,
            "date": msg.date.isoformat() if msg.date else None,
            "text": msg.text,
            "has_image": bool(img_filename),
            "image_filename": img_filename
        })

    return messages


In [7]:
async def scrape_channel(channel_username, limit=500):
    await client.start()
    entity = await client.get_entity(channel_username)
    messages = []

    today = datetime.utcnow().strftime("%Y-%m-%d")
    img_folder = os.path.join(DATA_PATH, today, channel_username, "images")
    os.makedirs(img_folder, exist_ok=True)

    async for msg in client.iter_messages(entity, limit=limit):
        img_filename = await download_media(msg, img_folder)
        messages.append({
            "message_id": msg.id,
            "date": msg.date.isoformat(),
            "text": msg.text,
            "has_image": bool(img_filename),
            "image_filename": img_filename
        })

    return messages


In [8]:
import asyncio  # make sure this is imported

async def scrape_all_channels():
    today = datetime.utcnow().strftime("%Y-%m-%d")
    out_dir = os.path.join(DATA_PATH, today)
    os.makedirs(out_dir, exist_ok=True)

    for channel in CHANNELS:
        filepath = os.path.join(out_dir, f"{channel}.json")

        # Skip if already scraped
        if os.path.exists(filepath):
            print(f"⏭️ Already scraped {channel}, skipping.")
            continue

        try:
            print(f"📥 Scraping {channel} (limit 500)...")
            messages = await scrape_channel(channel, limit=500)

            with open(filepath, "w", encoding="utf-8") as f:
                json.dump(messages, f, ensure_ascii=False, indent=2)

            print(f"✅ Scraped {len(messages)} messages from {channel}")

            # Add delay between channels to avoid resource locking
            await asyncio.sleep(5)

        except Exception as e:
            print(f"❌ Failed scraping {channel}: {e}")


In [9]:
await scrape_all_channels()


⏭️ Already scraped CheMed123, skipping.
⏭️ Already scraped lobelia4cosmetics, skipping.
📥 Scraping tikvahpharma (limit 500)...


Please enter your phone (or bot token):  +251987040409
Please enter the code you received:  20831
Please enter your password:  ········


Signed in successfully as M; remember to not break the ToS or you will risk an account ban!
✅ Scraped 500 messages from tikvahpharma
